In [ ]:
# 初始化组件
self.first_stage.initialize(ctx=ctx)
self.res_layers.initialize(ctx=ctx)
self.head.initialize(ctx=ctx)

# 加载特定深度的 Resnet 模型
if self.pretrained_base and not self.pretrained:
    if self.depth == 50:
        resnet2d = resnet50_v1b(pretrained=True)
    elif self.depth == 101:
        resnet2d = resnet101_v1b(pretrained=True)
    else:
        print('No such 2D pre-trained network of depth %d.' % (self.depth))

    weights2d = resnet2d.collect_params()
    if self.nonlocal_cfg is None:
        weights3d = self.collect_params()
    else:
        train_params_list = []
        raw_params = self.collect_params()
        for raw_name in raw_params.keys():
            if 'nonlocal' in raw_name:
                continue
            train_params_list.append(raw_name)
        init_patterns = '|'.join(train_params_list)
        weights3d = self.collect_params(init_patterns)
    assert len(weights2d.keys()) == len(weights3d.keys()), 'Number of parameters should be same.'

    dict2d = {}
    for key_id, key_name in enumerate(weights2d.keys()):
        dict2d[key_id] = key_name

    dict3d = {}
    for key_id, key_name in enumerate(weights3d.keys()):
        dict3d[key_id] = key_name

    dict_transform = {}
    for key_id, key_name in dict3d.items():
        dict_transform[dict2d[key_id]] = key_name

    cnt = 0
    for key2d, key3d in dict_transform.items():

        # 检查当前的 3D 权重参数名称 key3d 是否包含字符串 'conv'
        if 'conv' in key3d:

            # 从 3D 权重的形状中提取时间维度（temporal_dim）。在 I3D 网络中，时间维度通常是第三个维度（索引为 2）
            temporal_dim = weights3d[key3d].shape[2]

            # 使用 nd.expand_dims 函数将 2D 权重 weights2d[key2d] 的数据扩展一个维度。
            temporal_2d = nd.expand_dims(weights2d[key2d].data(), axis=2)

            # 使用 nd.broadcast_to 函数将扩展后的 2D 权重 temporal_2d 广播到目标形状 [0, 0, temporal_dim, 0, 0]。
            # 0 表示保留原来的维度大小，temporal_dim 表示在第三维（时间维度）上扩展到相应的大小。
            # shape 中有 5 个参数的原因是因为 I3D 网络的权重通常是五维的，代表了：批量大小 (batch size)、通道数 (channels)、时间维度 (temporal dimension)、高度 (height)、宽度 (width)。
            inflated_2d = nd.broadcast_to(temporal_2d, shape=[0, 0, temporal_dim, 0, 0]) / temporal_dim

            #  使用 assert 语句确认扩展后的 2D 权重 inflated_2d 的形状与目标 3D 权重 weights3d[key3d] 的形状相同。
            assert inflated_2d.shape == weights3d[key3d].shape, 'the shape of %s and %s does not match. ' % (key2d, key3d)

            # 这是一个逐个更新的过程，每次循环只处理一个权重参数。
            weights3d[key3d].set_data(inflated_2d)

            # cnt += 1 是用来记录已经成功迁移或更新的权重参数的数量。
            cnt += 1
            print('%s is done with shape: ' % (key3d), weights3d[key3d].shape)

    assert cnt == len(weights2d.keys()), 'Not all parameters have been ported, check the initialization.'